# SQLSage GRPO training (Colab)

**Person 3 — Meta OpenEnv Hackathon 2026.** Use an **A100** runtime. Set secrets: `WANDB_API_KEY`, `HF_TOKEN`, and your Space URL as `SQLSAGE_ENV_URL`.

This notebook is a skeleton: copy cells into a fresh Colab or run here after installing deps.

In [ ]:
# Cell 1 — installs (Colab)
# %pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
# %pip install -q trl openenv-core wandb transformers datasets accelerate huggingface_hub requests psycopg2-binary
import torch
assert torch.cuda.is_available(), "Switch Runtime → GPU A100"
print(torch.cuda.get_device_name(0))

In [ ]:
# Cell 2 — wandb + env URL
import os
import wandb
os.environ.setdefault("SQLSAGE_ENV_URL", "https://YOUR-HF-SPACE.hf.space")
wandb.login()  # or set WANDB_API_KEY in Colab secrets
run = wandb.init(project="sqlsage-grpo", name="colab-grpo-sqlsage", reinit=True)
print("Logging to", run.url)

In [ ]:
# Cell 3 — model (Unsloth + Qwen 1.5B) — enable when training for real
# from unsloth import FastLanguageModel
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name="Qwen/Qwen2.5-1.5B-Instruct",
#     max_seq_length=2048,
#     load_in_4bit=True,
#     dtype=None,
# )
# model = FastLanguageModel.get_peft_model(
#     model, r=16,
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
#     lora_alpha=16, lora_dropout=0, bias="none",
#     use_gradient_checkpointing="unsloth",
# )
print("Uncomment Cell 3 when ready to load Qwen + LoRA.")

In [ ]:
# Cell 4 — reward via HTTP env (smoke). Full GRPO: wire TRL GRPOTrainer `reward_funcs` to batched rollouts.
import requests
BASE = os.environ["SQLSAGE_ENV_URL"].rstrip("/")
r = requests.post(BASE + "/reset", json={}, timeout=120)
r.raise_for_status()
obs = r.json()["observation"]
body = {"action": {"action": "push_filter", "rewritten_query": obs["original_query"]}}
r2 = requests.post(BASE + "/step", json=body, timeout=600)
r2.raise_for_status()
print("reward", r2.json().get("reward"), "done", r2.json().get("done"))

In [ ]:
# Cell 5 — GRPOTrainer (reference §6.2). Fill in dataset + reward before running.
# from trl import GRPOTrainer, GRPOConfig
# from sqlsage.training.config import default_grpo_config
# cfg = GRPOConfig(**default_grpo_config())
# trainer = GRPOTrainer(model=model, processing_class=tokenizer, train_dataset=ds, reward_funcs=[...], args=cfg)
# trainer.train()
# model.save_pretrained_merged("sqlsage-trained", tokenizer, save_method="merged_16bit")
print("Uncomment and complete Cell 5 after dataset + reward wiring.")